# AI-Powered Fake News Detection Using Text Classification
**Summer Internship Project — AI & ML (2026)**

Pipeline built from scratch: preprocessing → feature extraction (BoW/TF-IDF) → model training (KNN, Logistic Regression, Random Forest, MLP) → evaluation.

Dataset: ISOT Fake News Dataset (`True.csv` + `Fake.csv`)


## Week 1 — Data Loading & Cleaning

### 1.1 Load and label the data
The dataset comes as two separate files. We load both, tag each with its label, and merge them into one DataFrame.

In [1]:
import pandas as pd
import numpy as np
import re

true_df = pd.read_csv('../data/True.csv')
fake_df = pd.read_csv('../data/Fake.csv')

true_df['label'] = 1   # 1 = real
fake_df['label'] = 0   # 0 = fake

df = pd.concat([true_df, fake_df], axis=0, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print("Total articles:", df.shape[0])
print(df['label'].value_counts().rename({1: 'real', 0: 'fake'}))
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/True.csv'

### 1.2 Basic inspection
Check for missing values and combine `title` + `text` into a single field (more signal for the model).

In [2]:
print("Missing values:\n", df.isnull().sum())

# Combine title + text into one field
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df = df[df['content'].str.strip().str.len() > 0].reset_index(drop=True)
print("\nShape after dropping empty rows:", df.shape)

NameError: name 'df' is not defined

### 1.3 Clean the text
Steps:
1. Strip the `"CITY (Reuters) -"` dateline pattern (dataset artifact that would leak the label).
2. Lowercase.
3. Remove URLs, punctuation, numbers.
4. Remove stopwords (using a manually defined list — no NLTK download needed, keeps this "from scratch").
5. Tokenize manually (`.split()` on cleaned text).

In [3]:
# Manually defined stopword list (from-scratch requirement — no external NLTK download)
STOPWORDS = set("""
a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does
doesn't doing don't down during each few for from further had hadn't has hasn't have
haven't having he he'd he'll he's her here here's hers herself him himself his how
how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out
over own same shan't she she'd she'll she's should shouldn't so some such than that
that's the their theirs them themselves then there there's these they they'd they'll
they're they've this those through to too under until up very was wasn't we we'd we'll
we're we've were weren't what what's when when's where where's which while who who's
whom why why's with won't would wouldn't you you'd you'll you're you've your yours
yourself yourselves
""".split())

REUTERS_TAG = re.compile(r'^[A-Z\s]+\(Reuters\)\s*-\s*')
URL_RE = re.compile(r'http\S+|www\.\S+')

def clean_text(text):
    text = REUTERS_TAG.sub('', text)          # strip leaking dateline artifact
    text = URL_RE.sub(' ', text)               # remove URLs
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)      # remove punctuation & numbers
    text = re.sub(r'\s+', ' ', text).strip()   # collapse whitespace
    return text

def tokenize(text):
    return [w for w in text.split() if w not in STOPWORDS and len(w) > 2]

df['clean_text'] = df['content'].apply(clean_text)
df['tokens'] = df['clean_text'].apply(tokenize)
df['final_text'] = df['tokens'].apply(lambda toks: ' '.join(toks))

df[['content', 'clean_text', 'final_text']].head(3)

NameError: name 'df' is not defined

In [4]:
# Drop any rows that became empty after cleaning
df = df[df['final_text'].str.len() > 0].reset_index(drop=True)
print("Final shape after cleaning:", df.shape)

# Save the cleaned dataset so Week 2+ can start from here without re-running cleaning
df[['final_text', 'label']].to_csv('../data/cleaned_data.csv', index=False)
print("Saved cleaned_data.csv")

NameError: name 'df' is not defined